## Setup Project

In [1]:
from pathlib import Path
from datetime import datetime
from urllib.parse import urlparse
from difflib import SequenceMatcher
import ipaddress
import json
import re

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

direktori_aktif = Path.cwd()

if direktori_aktif.name.lower() == "notebooks":
    direktori_project = direktori_aktif.parent
else:
    direktori_project = direktori_aktif

direktori_raw = direktori_project / "data" / "raw"
direktori_processed = direktori_project / "data" / "processed"
direktori_multi = direktori_processed / "multi_dataset"
direktori_intelligence = direktori_project / "data" / "intelligence"
direktori_outputs = direktori_project / "reports" / "outputs"

for folder in [
    direktori_raw,
    direktori_processed,
    direktori_multi,
    direktori_intelligence,
    direktori_outputs,
]:
    folder.mkdir(parents=True, exist_ok=True)

lokasi_phiusiil = direktori_raw / "PhiUSIIL_Phishing_URL_Dataset.csv"
lokasi_dataset_tambahan = direktori_multi / "dataset_tambahan_phreshphish_deepurlbench_tranco_clean.csv"
lokasi_fitur_v2 = direktori_outputs / "daftar_fitur_intelligence_v2.json"

print("Direktori project:", direktori_project)
print("Dataset PhiUSIIL:", lokasi_phiusiil.exists())
print("Dataset tambahan:", lokasi_dataset_tambahan.exists())
print("Daftar fitur V2:", lokasi_fitur_v2.exists())

Direktori project: C:\Users\ASUS\PHISHING
Dataset PhiUSIIL: True
Dataset tambahan: True
Daftar fitur V2: True


## Helper Standardisasi URL dan Label

In [9]:
def cari_kolom(data, kandidat):
    kolom_lower = {kolom.lower(): kolom for kolom in data.columns}

    for nama in kandidat:
        if nama.lower() in kolom_lower:
            return kolom_lower[nama.lower()]

    return None


def bersihkan_url(url):
    if pd.isna(url):
        return ""

    url = str(url).strip()
    url = url.replace("\x00", "")
    url = re.sub(r"\s+", "", url)

    return url


def ambil_domain(url):
    url = bersihkan_url(url)

    if not url:
        return ""

    try:
        url_parse = url if re.match(r"^https?://", url, flags=re.I) else "http://" + url
        parsed = urlparse(url_parse)
        domain = parsed.netloc.lower()
        domain = domain.split("@")[-1]
        domain = domain.split(":")[0]
        domain = domain.replace("www.", "", 1)

        return domain
    except Exception:
        return ""


def mapping_target_umum(label):
    teks = str(label).strip().lower()

    label_aman = {
        "benign",
        "legit",
        "legitimate",
        "safe",
        "good",
        "normal",
        "clean",
        "tranco_legitimate",
    }

    label_berisiko = {
        "phish",
        "phishing",
        "malware",
        "mal",
        "malicious",
        "defacement",
        "suspicious",
        "bad",
        "threat",
    }

    if teks in label_aman:
        return 0

    if teks in label_berisiko:
        return 1

    if teks == "0":
        return 0

    if teks == "1":
        return 1

    return np.nan


def mapping_target_phiusiil(label):
    teks = str(label).strip().lower()

    if teks == "1":
        return 0

    if teks == "0":
        return 1

    return np.nan


def standar_dataset_url(data, nama_dataset, sumber_data, split="unknown"):
    kolom_url = cari_kolom(data, ["url", "URL", "Url", "link", "Link"])
    kolom_target = cari_kolom(data, ["target_phishing", "target", "label", "Label", "class", "type", "status"])

    if kolom_url is None:
        raise ValueError(f"Kolom URL tidak ditemukan. Kolom tersedia: {list(data.columns)}")

    if kolom_target is None:
        raise ValueError(f"Kolom target tidak ditemukan. Kolom tersedia: {list(data.columns)}")

    hasil = pd.DataFrame()
    hasil["url"] = data[kolom_url].apply(bersihkan_url)
    hasil["domain"] = hasil["url"].apply(ambil_domain)
    hasil["original_label"] = data[kolom_target].astype(str)

    if nama_dataset.lower() == "phiusiil":
        hasil["target_phishing"] = hasil["original_label"].apply(mapping_target_phiusiil)
    else:
        hasil["target_phishing"] = hasil["original_label"].apply(mapping_target_umum)

    hasil["dataset_name"] = nama_dataset
    hasil["sumber_data"] = sumber_data
    hasil["split"] = split
    hasil["tanggal_diproses"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    hasil = hasil[hasil["url"] != ""]
    hasil = hasil[hasil["domain"] != ""]
    hasil = hasil.dropna(subset=["target_phishing"])
    hasil["target_phishing"] = hasil["target_phishing"].astype(int)

    return hasil.reset_index(drop=True)

## Baca PhiUSIIL dan Dataset Tambahan

In [10]:
data_phiusiil_raw = pd.read_csv(lokasi_phiusiil, low_memory=False)
data_tambahan = pd.read_csv(lokasi_dataset_tambahan, dtype=str, low_memory=False)

data_phiusiil = standar_dataset_url(
    data=data_phiusiil_raw,
    nama_dataset="PhiUSIIL",
    sumber_data="local_phiusiil_raw",
    split="original",
)

data_tambahan["target_phishing"] = data_tambahan["target_phishing"].astype(int)

kolom_standar = [
    "url",
    "domain",
    "original_label",
    "target_phishing",
    "dataset_name",
    "sumber_data",
    "split",
    "tanggal_diproses",
]

data_tambahan = data_tambahan[kolom_standar].copy()

data_url_multi = pd.concat(
    [
        data_phiusiil[kolom_standar],
        data_tambahan[kolom_standar],
    ],
    ignore_index=True,
)

data_url_multi["url"] = data_url_multi["url"].astype(str).str.strip()
data_url_multi["domain"] = data_url_multi["domain"].astype(str).str.strip().str.lower()
data_url_multi = data_url_multi[data_url_multi["url"] != ""]
data_url_multi = data_url_multi[data_url_multi["domain"] != ""]
data_url_multi = data_url_multi[data_url_multi["target_phishing"].isin([0, 1])]

jumlah_label_per_url = (
    data_url_multi
    .groupby("url")["target_phishing"]
    .nunique()
    .reset_index(name="jumlah_label_unik")
)

url_konflik = jumlah_label_per_url[
    jumlah_label_per_url["jumlah_label_unik"] > 1
]["url"].tolist()

data_konflik = data_url_multi[data_url_multi["url"].isin(url_konflik)].copy()
data_url_multi_clean = data_url_multi[~data_url_multi["url"].isin(url_konflik)].copy()
data_url_multi_clean = data_url_multi_clean.drop_duplicates(subset=["url"]).reset_index(drop=True)

lokasi_konflik_multi = direktori_multi / "dataset_multi_v5_konflik_label_url.csv"
lokasi_url_multi_clean = direktori_multi / "dataset_url_multi_v5_clean.csv"

data_konflik.to_csv(lokasi_konflik_multi, index=False, encoding="utf-8")
data_url_multi_clean.to_csv(lokasi_url_multi_clean, index=False, encoding="utf-8")

print("Dataset URL multi V5 berhasil disiapkan.")
print("Ukuran PhiUSIIL:", data_phiusiil.shape)
print("Ukuran tambahan:", data_tambahan.shape)
print("Ukuran clean:", data_url_multi_clean.shape)
print("Jumlah konflik URL:", len(url_konflik))

display(data_url_multi_clean["dataset_name"].value_counts())
display(data_url_multi_clean["target_phishing"].value_counts())
display(data_url_multi_clean.head())

Dataset URL multi V5 berhasil disiapkan.
Ukuran PhiUSIIL: (235795, 8)
Ukuran tambahan: (739961, 8)
Ukuran clean: (975090, 8)
Jumlah konflik URL: 5


dataset_name
DeepURLBench    359819
PhreshPhish     279936
PhiUSIIL        235365
Tranco           99970
Name: count, dtype: int64

target_phishing
0    509023
1    466067
Name: count, dtype: int64

,url,domain,original_label,target_phishing,dataset_name,sumber_data,split,tanggal_diproses
0,https://www.southbankmosaics.com,southbankmosaics.com,1,0,PhiUSIIL,local_phiusiil_raw,original,2026-05-18 21:01:39
1,https://www.uni-mainz.de,uni-mainz.de,1,0,PhiUSIIL,local_phiusiil_raw,original,2026-05-18 21:01:39
2,https://www.voicefmradio.co.uk,voicefmradio.co.uk,1,0,PhiUSIIL,local_phiusiil_raw,original,2026-05-18 21:01:39
3,https://www.sfnmjournal.com,sfnmjournal.com,1,0,PhiUSIIL,local_phiusiil_raw,original,2026-05-18 21:01:39
4,https://www.rewildingargentina.org,rewildingargentina.org,1,0,PhiUSIIL,local_phiusiil_raw,original,2026-05-18 21:01:39


## Load Daftar Fitur dan Data

In [11]:
def baca_daftar_fitur(lokasi_file):
    with open(lokasi_file, "r", encoding="utf-8") as file:
        data = json.load(file)

    if isinstance(data, list):
        return data

    if isinstance(data, dict):
        for key in ["features", "daftar_fitur", "feature_names"]:
            if key in data and isinstance(data[key], list):
                return data[key]

    raise ValueError("Format daftar fitur tidak dikenali.")


daftar_fitur_v5 = baca_daftar_fitur(lokasi_fitur_v2)

lokasi_official = direktori_intelligence / "official_domains_global.csv"
lokasi_brand = direktori_intelligence / "brand_keywords_global.csv"
lokasi_suspicious = direktori_intelligence / "suspicious_keywords_global.csv"

data_official = pd.read_csv(lokasi_official)
data_brand = pd.read_csv(lokasi_brand)
data_suspicious = pd.read_csv(lokasi_suspicious)

official_domains = (
    data_official["domain"]
    .dropna()
    .astype(str)
    .str.lower()
    .str.strip()
    .unique()
    .tolist()
)

brand_keywords = (
    data_brand["keyword"]
    .dropna()
    .astype(str)
    .str.lower()
    .str.strip()
    .unique()
    .tolist()
)

brand_map = {}
for _, row in data_brand.iterrows():
    keyword = str(row.get("keyword", "")).strip().lower()
    brand = str(row.get("brand", keyword)).strip()

    if keyword:
        brand_map[keyword] = brand

suspicious_weights = {}
for _, row in data_suspicious.iterrows():
    keyword = str(row.get("keyword", "")).strip().lower()

    try:
        bobot = int(row.get("bobot", 1))
    except Exception:
        bobot = 1

    if keyword:
        suspicious_weights[keyword] = bobot

print("Jumlah fitur V5:", len(daftar_fitur_v5))
print("Jumlah domain resmi:", len(official_domains))
print("Jumlah keyword brand:", len(brand_keywords))
print("Jumlah suspicious keyword:", len(suspicious_weights))

Jumlah fitur V5: 49
Jumlah domain resmi: 50
Jumlah keyword brand: 45
Jumlah suspicious keyword: 30


## Feature Extraction URL dan Intelligence

In [12]:
def cek_ip_domain(domain):
    try:
        ipaddress.ip_address(domain)
        return 1
    except Exception:
        return 0


def ambil_tld(domain):
    bagian = str(domain).split(".")

    if len(bagian) < 2:
        return ""

    return bagian[-1].lower()


def hitung_subdomain(domain):
    bagian = str(domain).split(".")

    if len(bagian) <= 2:
        return 0

    return max(0, len(bagian) - 2)


def cek_domain_resmi(domain):
    domain = str(domain).lower().strip()

    for official in official_domains:
        if domain == official or domain.endswith("." + official):
            return 1

    return 0


def cek_brand(url, domain):
    teks = f"{url} {domain}".lower()
    brand_terdeteksi = []

    for keyword in brand_keywords:
        if keyword and keyword in teks:
            brand_terdeteksi.append(keyword)

    if not brand_terdeteksi:
        return 0, ""

    brand_utama = brand_terdeteksi[0]
    return 1, brand_map.get(brand_utama, brand_utama)


def cek_suspicious_keywords(url):
    teks = str(url).lower()
    keyword_terdeteksi = []
    skor = 0

    for keyword, bobot in suspicious_weights.items():
        if keyword in teks:
            keyword_terdeteksi.append(keyword)
            skor += bobot

    return len(keyword_terdeteksi), skor


def ubah_digit_mirip(teks):
    tabel = str.maketrans({
        "0": "o",
        "1": "i",
        "3": "e",
        "4": "a",
        "5": "s",
        "7": "t",
    })

    return str(teks).translate(tabel)


def cek_lookalike(domain, is_official):
    if is_official:
        return 0, 0.0

    domain_bersih = re.sub(r"[^a-z0-9]", "", str(domain).lower())
    domain_bersih = ubah_digit_mirip(domain_bersih)

    skor_terbaik = 0.0

    for keyword in brand_keywords:
        keyword_bersih = re.sub(r"[^a-z0-9]", "", keyword.lower())

        if len(keyword_bersih) <= 2:
            continue

        skor = SequenceMatcher(None, domain_bersih, keyword_bersih).ratio()
        skor_terbaik = max(skor_terbaik, skor)

        if keyword_bersih in domain_bersih and keyword_bersih != domain_bersih:
            skor_terbaik = max(skor_terbaik, 1.0)

    return int(skor_terbaik >= 0.82), round(float(skor_terbaik), 4)


def ekstrak_fitur_satu_url(url):
    url = bersihkan_url(url)
    domain = ambil_domain(url)
    tld = ambil_tld(domain)

    jumlah_huruf = sum(karakter.isalpha() for karakter in url)
    jumlah_digit = sum(karakter.isdigit() for karakter in url)
    jumlah_spesial = sum(not karakter.isalnum() for karakter in url)

    panjang_url = len(url)
    panjang_domain = len(domain)

    fitur = {
        "URLLength": panjang_url,
        "DomainLength": panjang_domain,
        "TLDLength": len(tld),
        "NoOfSubDomain": hitung_subdomain(domain),
        "IsHTTPS": int(url.lower().startswith("https://")),
        "IsDomainIP": cek_ip_domain(domain),
        "NoOfLettersInURL": jumlah_huruf,
        "NoOfDegitsInURL": jumlah_digit,
        "NoOfDigitsInURL": jumlah_digit,
        "NoOfOtherSpecialCharsInURL": jumlah_spesial,
        "SpacialCharRatioInURL": jumlah_spesial / panjang_url if panjang_url else 0,
        "SpecialCharRatioInURL": jumlah_spesial / panjang_url if panjang_url else 0,
        "LetterRatioInURL": jumlah_huruf / panjang_url if panjang_url else 0,
        "DegitRatioInURL": jumlah_digit / panjang_url if panjang_url else 0,
        "DigitRatioInURL": jumlah_digit / panjang_url if panjang_url else 0,
        "NoOfEqualsInURL": url.count("="),
        "NoOfQMarkInURL": url.count("?"),
        "NoOfAmpersandInURL": url.count("&"),
        "NoOfSlashInURL": url.count("/"),
        "NoOfDotInURL": url.count("."),
        "NoOfDashInURL": url.count("-"),
        "NoOfAtInURL": url.count("@"),
    }

    for kolom in daftar_fitur_v5:
        if kolom.startswith("TLD_"):
            fitur[kolom] = 0

    kolom_tld = f"TLD_{tld}"

    if kolom_tld in fitur:
        fitur[kolom_tld] = 1
    elif "TLD_lainnya" in fitur:
        fitur["TLD_lainnya"] = 1

    is_official = cek_domain_resmi(domain)
    brand_keyword_detected, brand_detected = cek_brand(url, domain)
    suspicious_count, suspicious_score = cek_suspicious_keywords(url)
    lookalike_detected, lookalike_score = cek_lookalike(domain, is_official)

    fitur.update({
        "is_official_domain": is_official,
        "brand_keyword_detected": brand_keyword_detected,
        "brand_but_not_official": int(brand_keyword_detected == 1 and is_official == 0),
        "suspicious_keyword_count": suspicious_count,
        "suspicious_keyword_score": suspicious_score,
        "lookalike_brand_detected": lookalike_detected,
        "lookalike_score": lookalike_score,
        "uses_punycode": int("xn--" in domain),
        "uses_digit_substitution": int(any(karakter.isdigit() for karakter in domain)),
        "hyphen_count": domain.count("-"),
    })

    hasil = {fitur_nama: fitur.get(fitur_nama, 0) for fitur_nama in daftar_fitur_v5}

    return hasil

## Ekstrak Fitur Multi Dataset

In [13]:
ukuran_chunk = 50_000

lokasi_dataset_training_v5 = direktori_multi / "dataset_training_multi_v5.csv"
lokasi_fitur_training_v5 = direktori_multi / "fitur_training_multi_v5.csv"
lokasi_target_training_v5 = direktori_multi / "target_training_multi_v5.csv"
lokasi_daftar_fitur_v5 = direktori_outputs / "daftar_fitur_multi_dataset_v5.json"

for lokasi in [
    lokasi_dataset_training_v5,
    lokasi_fitur_training_v5,
    lokasi_target_training_v5,
]:
    if lokasi.exists():
        lokasi.unlink()

metadata_cols = [
    "url",
    "domain",
    "original_label",
    "target_phishing",
    "dataset_name",
    "sumber_data",
    "split",
]

tulis_header_dataset = True
tulis_header_fitur = True
tulis_header_target = True

for start in tqdm(range(0, len(data_url_multi_clean), ukuran_chunk), desc="Ekstraksi fitur V5"):
    end = start + ukuran_chunk
    data_chunk = data_url_multi_clean.iloc[start:end].copy()

    fitur_chunk = pd.DataFrame(
        [ekstrak_fitur_satu_url(url) for url in data_chunk["url"]]
    )

    fitur_chunk = fitur_chunk[daftar_fitur_v5].copy()
    target_chunk = data_chunk[["target_phishing"]].copy()
    dataset_chunk = pd.concat(
        [
            data_chunk[metadata_cols].reset_index(drop=True),
            fitur_chunk.reset_index(drop=True),
        ],
        axis=1,
    )

    dataset_chunk.to_csv(
        lokasi_dataset_training_v5,
        mode="a",
        header=tulis_header_dataset,
        index=False,
        encoding="utf-8",
    )

    fitur_chunk.to_csv(
        lokasi_fitur_training_v5,
        mode="a",
        header=tulis_header_fitur,
        index=False,
        encoding="utf-8",
    )

    target_chunk.to_csv(
        lokasi_target_training_v5,
        mode="a",
        header=tulis_header_target,
        index=False,
        encoding="utf-8",
    )

    tulis_header_dataset = False
    tulis_header_fitur = False
    tulis_header_target = False

with open(lokasi_daftar_fitur_v5, "w", encoding="utf-8") as file:
    json.dump(daftar_fitur_v5, file, indent=4, ensure_ascii=False)

print("Dataset training V5 berhasil dibuat.")
print("Dataset training:", lokasi_dataset_training_v5)
print("Fitur training:", lokasi_fitur_training_v5)
print("Target training:", lokasi_target_training_v5)
print("Daftar fitur:", lokasi_daftar_fitur_v5)

Ekstraksi fitur V5:   0%|          | 0/20 [00:00<?, ?it/s]

Dataset training V5 berhasil dibuat.
Dataset training: C:\Users\ASUS\PHISHING\data\processed\multi_dataset\dataset_training_multi_v5.csv
Fitur training: C:\Users\ASUS\PHISHING\data\processed\multi_dataset\fitur_training_multi_v5.csv
Target training: C:\Users\ASUS\PHISHING\data\processed\multi_dataset\target_training_multi_v5.csv
Daftar fitur: C:\Users\ASUS\PHISHING\reports\outputs\daftar_fitur_multi_dataset_v5.json


## Validasi Dataset Training

In [14]:
data_validasi_v5 = pd.read_csv(
    lokasi_dataset_training_v5,
    usecols=[
        "url",
        "domain",
        "target_phishing",
        "dataset_name",
        "original_label",
    ],
    low_memory=False,
)

validasi_v5 = pd.DataFrame([
    {"metrik": "jumlah_data", "nilai": len(data_validasi_v5)},
    {"metrik": "jumlah_url_unik", "nilai": data_validasi_v5["url"].nunique()},
    {"metrik": "jumlah_domain_unik", "nilai": data_validasi_v5["domain"].nunique()},
    {"metrik": "jumlah_duplikat_url", "nilai": int(data_validasi_v5.duplicated(subset=["url"]).sum())},
    {"metrik": "jumlah_null_url", "nilai": int(data_validasi_v5["url"].isna().sum())},
    {"metrik": "jumlah_null_domain", "nilai": int(data_validasi_v5["domain"].isna().sum())},
    {"metrik": "jumlah_target_aman", "nilai": int((data_validasi_v5["target_phishing"] == 0).sum())},
    {"metrik": "jumlah_target_berisiko", "nilai": int((data_validasi_v5["target_phishing"] == 1).sum())},
])

ringkasan_dataset_v5 = (
    data_validasi_v5
    .groupby(["dataset_name", "target_phishing"])
    .size()
    .reset_index(name="jumlah_data")
)

ringkasan_label_v5 = (
    data_validasi_v5["original_label"]
    .value_counts()
    .reset_index()
)

ringkasan_label_v5.columns = ["original_label", "jumlah_data"]

lokasi_validasi_v5 = direktori_outputs / "validasi_dataset_training_multi_v5.csv"
lokasi_ringkasan_dataset_v5 = direktori_outputs / "ringkasan_dataset_training_multi_v5.csv"
lokasi_ringkasan_label_v5 = direktori_outputs / "ringkasan_label_training_multi_v5.csv"

validasi_v5.to_csv(lokasi_validasi_v5, index=False, encoding="utf-8")
ringkasan_dataset_v5.to_csv(lokasi_ringkasan_dataset_v5, index=False, encoding="utf-8")
ringkasan_label_v5.to_csv(lokasi_ringkasan_label_v5, index=False, encoding="utf-8")

print("Validasi dataset training selesai.")
display(validasi_v5)
display(ringkasan_dataset_v5)
display(ringkasan_label_v5.head(20))

Validasi dataset training selesai.


,metrik,nilai
0,jumlah_data,975090
1,jumlah_url_unik,975090
2,jumlah_domain_unik,773659
3,jumlah_duplikat_url,0
4,jumlah_null_url,0
5,jumlah_null_domain,0
6,jumlah_target_aman,509023
7,jumlah_target_berisiko,466067


,dataset_name,target_phishing,jumlah_data
0,DeepURLBench,0,119936
1,DeepURLBench,1,239883
2,PhiUSIIL,0,134848
3,PhiUSIIL,1,100517
4,PhreshPhish,0,154269
5,PhreshPhish,1,125667
6,Tranco,0,99970


,original_label,jumlah_data
0,benign,274205
1,1,134848
2,phish,125667
3,phishing,119994
4,malware,119889
5,0,100517
6,tranco_legitimate,99970


## Metadata Final Feature Extraction

In [15]:
metadata_v5 = {
    "nama_notebook": "14_feature_extraction_multi_dataset_v5.ipynb",
    "nama_tahap": "Feature Extraction Multi Dataset V5",
    "status": "selesai",
    "tujuan": "Membuat dataset training V5 dari PhiUSIIL, PhreshPhish, DeepURLBench, dan Tranco.",
    "jumlah_data": int(len(data_validasi_v5)),
    "jumlah_fitur": int(len(daftar_fitur_v5)),
    "file_input": {
        "phiusiil": str(lokasi_phiusiil),
        "dataset_tambahan_clean": str(lokasi_dataset_tambahan),
        "daftar_fitur_v2": str(lokasi_fitur_v2),
    },
    "file_output": {
        "dataset_url_multi_clean": str(lokasi_url_multi_clean),
        "dataset_training_v5": str(lokasi_dataset_training_v5),
        "fitur_training_v5": str(lokasi_fitur_training_v5),
        "target_training_v5": str(lokasi_target_training_v5),
        "daftar_fitur_v5": str(lokasi_daftar_fitur_v5),
        "validasi_v5": str(lokasi_validasi_v5),
        "ringkasan_dataset_v5": str(lokasi_ringkasan_dataset_v5),
        "ringkasan_label_v5": str(lokasi_ringkasan_label_v5),
    },
    "catatan": [
        "Target masih bernama target_phishing agar kompatibel dengan pipeline lama.",
        "Secara konsep target 1 berarti URL berisiko karena DeepURLBench memasukkan malware.",
        "Tahap berikutnya adalah training model V5 dan evaluasi multi-dataset.",
        "File besar jangan dipush langsung ke GitHub.",
    ],
    "tanggal_selesai": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
}

lokasi_metadata_v5 = direktori_outputs / "metadata_feature_extraction_multi_dataset_v5.json"

with open(lokasi_metadata_v5, "w", encoding="utf-8") as file:
    json.dump(metadata_v5, file, indent=4, ensure_ascii=False)

catatan_v5 = f"""
CATATAN FINAL FEATURE EXTRACTION MULTI DATASET V5

Notebook:
14_feature_extraction_multi_dataset_v5.ipynb

Output utama:
{lokasi_dataset_training_v5}

Jumlah data:
{len(data_validasi_v5)}

Jumlah fitur:
{len(daftar_fitur_v5)}

Tahap berikutnya:
15_training_model_multi_dataset_v5.ipynb

Catatan:
Dataset V5 sudah menggabungkan PhiUSIIL, PhreshPhish, DeepURLBench, dan Tranco.
Target 1 berarti URL berisiko, termasuk phishing dan malware.
"""

lokasi_catatan_v5 = direktori_outputs / "catatan_final_feature_extraction_multi_dataset_v5.txt"
lokasi_catatan_v5.write_text(catatan_v5, encoding="utf-8")

print(catatan_v5)
print("Metadata disimpan:", lokasi_metadata_v5)
print("Catatan disimpan:", lokasi_catatan_v5)


CATATAN FINAL FEATURE EXTRACTION MULTI DATASET V5

Notebook:
14_feature_extraction_multi_dataset_v5.ipynb

Output utama:
C:\Users\ASUS\PHISHING\data\processed\multi_dataset\dataset_training_multi_v5.csv

Jumlah data:
975090

Jumlah fitur:
49

Tahap berikutnya:
15_training_model_multi_dataset_v5.ipynb

Catatan:
Dataset V5 sudah menggabungkan PhiUSIIL, PhreshPhish, DeepURLBench, dan Tranco.
Target 1 berarti URL berisiko, termasuk phishing dan malware.

Metadata disimpan: C:\Users\ASUS\PHISHING\reports\outputs\metadata_feature_extraction_multi_dataset_v5.json
Catatan disimpan: C:\Users\ASUS\PHISHING\reports\outputs\catatan_final_feature_extraction_multi_dataset_v5.txt
